# Trích xuất Image Embeddings cho Đồ án Tốt nghiệp (DATN)
Notebook này được thiết kế để chạy trên Kaggle nhằm tải song song các hình ảnh sản phẩm từ `image_url` và dùng mô hình học sâu **CLIP Image Encoder (ViT-B/32)** trên GPU để trích xuất đặc trưng hình ảnh.

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q polars transformers pillow tqdm requests pyarrow

In [ ]:
# 2. Import các thư viện
import os
import io
import requests
import numpy as np
import polars as pl
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPVisionModel, CLIPProcessor

# Tránh lỗi khi đọc ảnh bị khuyết hoặc hỏng một phần
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Thiết lập thiết bị chạy GPU/CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# 3. Cấu hình đường dẫn
INPUT_DIR = "/kaggle/input/datn-stream-subset"
OUTPUT_DIR = "/kaggle/working"
IMAGE_DOWNLOAD_DIR = "/kaggle/working/downloaded_images"

# Tên mô hình đa phương thức để sử dụng nhánh vision
MODEL_NAME = "openai/clip-vit-base-patch32"

os.makedirs(IMAGE_DOWNLOAD_DIR, exist_ok=True)

items_path = os.path.join(INPUT_DIR, "items.parquet")
if not os.path.exists(items_path):
    # Fallback cho chạy local hoặc thử nghiệm
    items_path = "../data/items.parquet"
    INPUT_DIR = "../data"
    IMAGE_DOWNLOAD_DIR = "../data/downloaded_images"
    os.makedirs(IMAGE_DOWNLOAD_DIR, exist_ok=True)
    
print(f"Loading items from: {items_path}")

In [ ]:
# 4. Đọc dữ liệu bằng Polars
df_items = pl.read_parquet(items_path)
print(f"Tổng số sản phẩm: {len(df_items)}")
df_items.head(3)

In [ ]:
# 5. Định nghĩa hàm tải ảnh đa luồng (Multi-threading Downloader)
# Do tải ảnh qua URL gặp độ trễ mạng, ta dùng ThreadPoolExecutor để tăng tốc gấp nhiều lần.

def download_single_image(item_id, url, save_dir):
    if not url:
        return item_id, False, "Missing URL"
    
    # Xác định đường dẫn lưu file ảnh dạng JPG
    file_path = os.path.join(save_dir, f"{item_id}.jpg")
    if os.path.exists(file_path):
        return item_id, True, "Already Exists"
        
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
        response = requests.get(url, timeout=10, headers=headers)
        if response.status_code == 200:
            # Đọc thử ảnh để kiểm tra tính hợp lệ
            img = Image.open(io.BytesIO(response.content))
            img.convert("RGB").save(file_path, "JPEG")
            return item_id, True, "Success"
        else:
            return item_id, False, f"HTTP {response.status_code}"
    except Exception as e:
        return item_id, False, str(e)

# Chuẩn bị dữ liệu tải
download_tasks = df_items.select(["item_id", "image_url"]).to_dicts()
print(f"Bắt đầu tải song song {len(download_tasks)} ảnh sản phẩm...")

success_count = 0
failed_count = 0
failed_log = {}

# Thực hiện tải với 32 luồng đồng thời (Max Workers)
with ThreadPoolExecutor(max_workers=32) as executor:
    futures = {
        executor.submit(download_single_image, task["item_id"], task["image_url"], IMAGE_DOWNLOAD_DIR): task
        for task in download_tasks
    }
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Đang tải hình ảnh"):
        item_id, success, message = future.result()
        if success:
            success_count += 1
        else:
            failed_count += 1
            failed_log[item_id] = message

print(f"Hoàn thành tải ảnh! Thành công: {success_count}, Thất bại: {failed_count}")
if failed_count > 0:
    print("Ví dụ 5 lỗi tải ảnh đầu tiên:", list(failed_log.items())[:5])

In [ ]:
# 6. Tạo PyTorch Dataset tải và tiền xử lý ảnh
class ProductImageDataset(Dataset):
    def __init__(self, df, image_dir, processor):
        self.df = df
        self.image_dir = image_dir
        self.processor = processor
        self.item_ids = df["item_id"].to_list()
        
    def __len__(self):
        return len(self.item_ids)
        
    def __getitem__(self, idx):
        item_id = self.item_ids[idx]
        file_path = os.path.join(self.image_dir, f"{item_id}.jpg")
        
        # Nếu ảnh tải lỗi hoặc không tồn tại, tạo ảnh đen kích thước 224x224 làm dự phòng
        try:
            if os.path.exists(file_path):
                img = Image.open(file_path).convert("RGB")
            else:
                img = Image.new("RGB", (224, 224), color=0)
        except Exception:
            img = Image.new("RGB", (224, 224), color=0)
            
        # Áp dụng CLIP Processor để resize, normalize ảnh
        processed = self.processor(images=img, return_tensors="pt")
        # processed['pixel_values'] có shape (1, 3, 224, 224), loại bỏ dimension 1 dư thừa
        pixel_values = processed["pixel_values"].squeeze(0)
        
        return item_id, pixel_values

In [ ]:
# 7. Khởi tạo mô hình CLIP Image Encoder và Processor
print(f"Khởi tạo CLIP Model: {MODEL_NAME}...")
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPVisionModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval() # Chuyển sang chế độ evaluation
print("Tải mô hình thành công.")

In [ ]:
# 8. Chạy suy luận (Inference Loop) để trích xuất đặc trưng
dataset = ProductImageDataset(df_items, IMAGE_DOWNLOAD_DIR, processor)
# Đặt num_workers=2 hoặc 4 để tăng tốc đọc đĩa trên Kaggle
dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

all_embeddings = []
item_ids_ordered = []

print("Bắt đầu trích xuất Image Embedding...")
with torch.no_grad():
    for batch_ids, pixel_values in tqdm(dataloader, desc="Trích xuất đặc trưng ảnh"):
        pixel_values = pixel_values.to(device)
        # Trích xuất vector nhúng đặc trưng thị giác từ CLIP Vision Model
        outputs = model(pixel_values)
        # outputs.pooler_output chứa representation vector của ảnh [Batch, 768]
        features = outputs.pooler_output.cpu().numpy()
        
        all_embeddings.append(features)
        item_ids_ordered.extend(batch_ids)

image_embeddings = np.vstack(all_embeddings)
print(f"Hoàn tất! Kích thước ma trận embeddings hình ảnh: {image_embeddings.shape}")

In [ ]:
# 9. Lưu kết quả ra file npy và metadata
emb_output_path = os.path.join(OUTPUT_DIR, "image_embeddings.npy")
np.save(emb_output_path, image_embeddings)
print(f"Đã lưu ma trận vector nhúng ảnh tại: {emb_output_path}")

# Lưu metadata liên kết thứ tự index -> item_id
df_metadata = pl.DataFrame({
    "index": list(range(len(item_ids_ordered))),
    "item_id": item_ids_ordered
})
meta_output_path = os.path.join(OUTPUT_DIR, "image_embedding_metadata.parquet")
df_metadata.write_parquet(meta_output_path)
print(f"Đã lưu metadata hình ảnh tại: {meta_output_path}")

In [ ]:
# 10. Kiểm tra nhanh tính chính xác
loaded_embeddings = np.load(emb_output_path)
print(f"Kiểm tra kích thước file tải lại: {loaded_embeddings.shape}")
assert np.allclose(image_embeddings, loaded_embeddings), "Dữ liệu lưu bị lỗi!"